# 05. Handoffs - 상태 기반 에이전트 전환

> Handoff는 한 에이전트가 다른 에이전트로 **흐름을 넘기는** 패턴이에요. `current_step` 상태 변수와 `@wrap_model_call` 만으로 가벼운 단계 전환을 만들어 봅니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Handoffs 패턴의 핵심 개념과 Supervisor 패턴과의 차이를 설명할 수 있어요
2. `current_step` 상태 변수와 `@wrap_model_call` 미들웨어를 사용해 단일 에이전트 Handoffs를 구현할 수 있어요
3. `tool_call_id` 페어링 규칙을 이해하고 올바른 `ToolMessage` 응답을 구성할 수 있어요
4. `STEP_CONFIG` 딕셔너리로 단계별 프롬프트와 도구를 동적으로 전환하는 패턴을 설계할 수 있어요

## 사전 지식

- Part 5 `create_agent` 기반 에이전트 생성
- Part 6 `@wrap_model_call` 미들웨어 기초
- 이전 노트북 `04-Multi-Agent-Supervisor.ipynb` — LLM 기반 Supervisor 패턴

## Handoffs 패턴 개요

이전 노트북에서 배운 Supervisor 패턴은 LLM이 매번 다음 에이전트를 결정해요. 반면 **Handoffs 패턴**은 에이전트 자신이 상태 변수를 보고 행동 방식을 바꾸는 설계예요.

> 💡 **핵심 정리**: Handoffs 패턴은 **공장의 컨베이어 벨트**와 같아요. 원재료(사용자 요청)가 1번 공정(보증 확인) → 2번 공정(문제 분류) → 3번 공정(해결 처리)을 순서대로 거쳐요. Supervisor처럼 "다음 누구?" 하고 매번 물어보는 게 아니라, `current_step` 변수가 "지금 2번 공정이야"라고 알려주면 그에 맞는 도구와 프롬프트가 자동으로 세팅돼요.

### 두 가지 구현 방식

| 방식 | 핵심 요소 | 비유 | 특징 |
|------|----------|------|------|
| **단일 에이전트 + 미들웨어** | `current_step` + `@wrap_model_call` | 만능 직원이 다른 모자를 쓰는 것 | 하나의 에이전트가 단계별로 행동 변경, 권장 방식 |
| **다중 서브그래프** | `transfer_to_*` 도구 + 서브그래프 | 진짜 다른 직원에게 업무를 넘기는 것 | 에이전트 간 명시적 제어권 이전, 복잡한 아키텍처 |

### 핵심 아이디어: 상태 변수로 행동을 바꾼다

```python
# current_step 값에 따라 다른 프롬프트와 도구를 사용해요
STEP_CONFIG = {
    "warranty_check": {"prompt": "보증 확인 전문가", "tools": [check_warranty]},
    "issue_classify": {"prompt": "문제 분류 전문가", "tools": [classify_issue]},
    "resolve":        {"prompt": "문제 해결 전문가", "tools": [create_ticket]},
}
```

> 🔑 **핵심 개념**: Handoffs 패턴에서 "누가 다음인가"를 결정하는 것은 별도 Supervisor가 아니에요. **상태에 저장된 `current_step` 값** 자체가 에이전트의 행동을 결정해요. 단계 전환은 `Command(update={"current_step": "next_step"}, goto=...)` 한 줄로 이루어져요.

### Supervisor 패턴과 비교

| 항목 | Supervisor 패턴 | Handoffs 패턴 |
|------|----------------|---------------|
| **라우팅 주체** | 별도 Supervisor LLM | 상태 변수 (`current_step`) |
| **전환 비용** | 매번 LLM 호출 (비용 발생) | 상태 값 변경만 (비용 없음) |
| **흐름 예측성** | 낮음 (LLM이 판단) | 높음 (규칙 기반, 결정론적) |
| **구조** | 여러 에이전트 | 하나의 에이전트 (권장) |
| **적합한 경우** | 복잡한 동적 라우팅 | 미리 정의된 워크플로우 |

### 전체 시스템 아키텍처 (단일 에이전트 방식)

```mermaid
flowchart TD
    U(["사용자 요청<br/>User Request"]) --> A
    A(["단일 에이전트<br/>Single Agent"]) --> MW
    MW(["미들웨어<br/>wrap_model_call"]) --> SC
    SC{"current_step<br/>확인"}
    SC -->|"warranty_check"| W(["보증 확인<br/>단계"])
    SC -->|"issue_classify"| I(["문제 분류<br/>단계"])
    SC -->|"resolve"| R(["해결 처리<br/>단계"])
    W -->|"Command(update)"| SC
    I -->|"Command(update)"| SC
    R --> FIN(["완료<br/>END"])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef decision fill:#fce8d4,stroke:#fd7e14,color:#7d3a04

    class U input
    class A,MW process
    class SC decision
    class W,I,R storage
    class FIN output
```

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 불러와요
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택 사항)
# ---------------------------------------------------
# 실행 흐름을 시각적으로 확인하고 싶다면 아래 주석을 해제하세요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "Multi-Agent-Handoffs"

## 모델 설정

LangChain V1의 `init_chat_model`로 모델을 초기화해요. 기본 모델은 비용 효율적인 `gpt-4o-mini`를 사용해요.

In [2]:
# ---------------------------------------------------
# 모델 초기화
# ---------------------------------------------------
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델로 변경하려면:
#   Anthropic: "anthropic:claude-sonnet-4-5"
#   Ollama 로컬: "ollama:llama3"
model = init_chat_model("openai:gpt-4o-mini")
# 모델 초기화 완료: gpt-4o-mini

## 1. ToolMessage 페어링 규칙

Handoffs 패턴에서 도구 호출 결과를 올바르게 전달하려면 `ToolMessage`와 `AIMessage(tool_calls)`의 페어링을 맞춰야 해요.

### tool_call_id 필수 매칭

LLM이 도구를 호출하면 각 호출에는 고유한 `tool_call_id`가 생성돼요. `ToolMessage`는 반드시 이 `tool_call_id`를 포함해야 해요.

```
AIMessage(
    tool_calls=[{"id": "call_abc123", "name": "transfer_to_sales", "args": {...}}]
)
    ↓ 반드시 페어링
ToolMessage(
    tool_call_id="call_abc123",  # 동일한 id!
    content="전환 완료"
)
```

> ⚠️ **자주 하는 실수**: `tool_call_id`를 누락하거나 다른 값을 사용하면 LLM이 도구 응답을 인식하지 못해요. 대화 이력이 깨지고 에이전트가 무한 루프에 빠질 수 있어요.

> 🔑 **핵심 개념**: 서브그래프 방식에서 에이전트 간 핸드오프 시 반드시 `AIMessage(tool_calls=...)` + `ToolMessage(tool_call_id=...)` 쌍을 함께 전달해야 해요. 이 쌍이 "제어권 이전 완료" 신호예요.

In [5]:
# ---------------------------------------------------
# ToolMessage 페어링 규칙 확인
# ---------------------------------------------------
# tool_call_id 매칭이 왜 중요한지 직접 확인해봐요
from langchain.messages import AIMessage, ToolMessage, HumanMessage

# 잘못된 예: tool_call_id 불일치
ai_msg = AIMessage(
    content="",
    tool_calls=[{"id": "call_abc123", "name": "transfer_to_sales", "args": {}}]
)
wrong_tool_msg = ToolMessage(
    tool_call_id="call_wrong_id",  # 불일치! 오류 발생
    content="전환 완료"
)

# 올바른 예: tool_call_id 일치
correct_tool_msg = ToolMessage(
    tool_call_id="call_abc123",  # AIMessage의 id와 동일
    content="전환 완료"
)

# === ToolMessage 페어링 규칙 ===
print(f"AIMessage tool_call_id: {ai_msg.tool_calls[0]['id']}")
print(f"잘못된 ToolMessage id:  {wrong_tool_msg.tool_call_id} -> 불일치!")
print(f"올바른 ToolMessage id:  {correct_tool_msg.tool_call_id} -> 일치!")
# tool_call_id가 일치해야 LLM이 도구 응답을 올바르게 인식해요

AIMessage tool_call_id: call_abc123
잘못된 ToolMessage id:  call_wrong_id -> 불일치!
올바른 ToolMessage id:  call_abc123 -> 일치!


## 2. 상태(State) 정의

Handoffs 패턴의 핵심은 `current_step` 상태 변수예요. 이 값이 에이전트의 행동을 결정해요.

### SupportState 구성

| 필드 | 타입 | 역할 |
|------|------|------|
| `messages` | `list[BaseMessage]` (add_messages) | 전체 대화 이력 (자동 누적) |
| `current_step` | `str` | 현재 워크플로우 단계 (미들웨어가 읽음) |
| `warranty_status` | `str` (NotRequired) | 보증 확인 결과 저장 |
| `issue_type` | `str` (NotRequired) | 문제 유형 분류 결과 저장 |

> 💡 **실무 팁**: `NotRequired[str]`은 해당 필드가 없어도 오류가 발생하지 않아요. 워크플로우 초기에는 `warranty_status`나 `issue_type`이 없지만, 단계가 진행되면서 채워져요. 이런 "점진적으로 채워지는" 상태 설계가 Handoffs 패턴의 특징이에요.

In [ ]:
from typing_extensions import TypedDict, NotRequired
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing import Annotated

class SupportState(TypedDict):
    """
    고객 지원 워크플로우의 공유 상태

    current_step 값에 따라 에이전트의 행동이 달라진다.
    warranty_status와 issue_type은 단계가 진행되면서 채워진다(handsoff)
    """

    messages: Annotated[list[BaseMessage], add_messages]
    current_step: str
    warrenty_status: NotRequired[str]
    


## 3. STEP_CONFIG - 단계별 설정 딕셔너리

`STEP_CONFIG`는 각 단계의 시스템 프롬프트, 도구, 의존성을 한 곳에 모아두는 설정 딕셔너리예요.

### 구조

```python
STEP_CONFIG = {
    "단계명": {
        "system_prompt": "이 단계에서 에이전트가 따를 지시사항",
        "tools": [이 단계에서 사용할 도구 목록],
        "next_step": "다음으로 이동할 단계명"  # 선택 사항
    }
}
```

> 💡 **핵심 정리**: `STEP_CONFIG`의 핵심은 **설정과 로직의 분리**예요. 새로운 단계를 추가하려면 딕셔너리에 항목 하나만 추가하면 돼요. 에이전트 로직 코드는 건드리지 않아도 됩니다. 이것이 유지보수하기 쉬운 설계예요.

> 💡 **실무 팁**: 실제 고객 지원 시스템에서는 `STEP_CONFIG`를 외부 설정 파일(YAML, JSON)로 관리하기도 해요. 비즈니스 로직 변경이 코드 수정 없이 가능해져요.

In [1]:
# ---------------------------------------------------
# 각 단계에서 사용할 도구(Tool) 정의
# ---------------------------------------------------
from langchain.tools import tool


@tool
def check_warranty(product_id: str) -> str:
    """제품 보증 상태를 확인해요. product_id로 조회해요."""
    # 실제 환경에서는 DB나 API를 호출하지만, 여기서는 시뮬레이션해요
    warranty_db = {
        "PRD001": "valid",
        "PRD002": "expired",
        "PRD003": "valid",
    }
    status = warranty_db.get(product_id, "unknown")
    return f"제품 {product_id}의 보증 상태: {status}"


@tool
def classify_issue(description: str) -> str:
    """고객 문제를 유형별로 분류해요. 설명을 분석해 카테고리를 반환해요."""
    # 간단한 키워드 기반 분류 (실제로는 LLM이 분류하도록 설계)
    description_lower = description.lower()
    if any(k in description_lower for k in ["화면", "버튼", "배터리", "충전", "하드웨어"]):
        return "hardware"
    elif any(k in description_lower for k in ["앱", "소프트웨어", "업데이트", "오류", "crash"]):
        return "software"
    elif any(k in description_lower for k in ["환불", "청구", "결제", "요금", "billing"]):
        return "billing"
    else:
        return "other"


@tool
def create_support_ticket(issue_type: str, description: str, priority: str = "normal") -> str:
    """고객 지원 티켓을 생성해요. 문제 유형, 설명, 우선순위를 입력받아요."""
    import uuid
    ticket_id = f"TICKET-{str(uuid.uuid4())[:8].upper()}"
    return (
        f"티켓 생성 완료: {ticket_id}\n"
        f"  유형: {issue_type}\n"
        f"  설명: {description}\n"
        f"  우선순위: {priority}\n"
        f"  예상 처리 시간: 24시간 이내"
    )


@tool
def provide_solution(issue_type: str, warranty_status: str) -> str:
    """문제 유형과 보증 상태에 따른 해결책을 제공해요."""
    if warranty_status == "valid":
        if issue_type == "hardware":
            return "보증 기간 내 하드웨어 문제: 무상 수리 또는 교환을 진행해 드려요."
        elif issue_type == "software":
            return "보증 기간 내 소프트웨어 문제: 공식 업데이트를 시도하고 복구 모드를 안내해 드려요."
        else:
            return "보증 기간 내 문제: 담당 팀이 48시간 내 연락드릴 예정이에요."
    else:
        return "보증 기간 만료: 유상 수리 서비스를 안내해 드려요. 가까운 서비스 센터를 방문해 주세요."

/Users/elixir/dev/lecture/hk-toss/10_langgraph/.venv/lib/python3.14/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# ---------------------------------------------------
# STEP_CONFIG: 단계별 프롬프트 + 도구 + 의존성 매핑
# ---------------------------------------------------
# 새 단계 추가 시 여기에만 항목을 추가하면 돼요
STEP_CONFIG = {
    # 1단계: 보증 확인
    "warranty_check": {
        "system_prompt": (
            "당신은 고객 지원팀의 보증 확인 전문가예요. "
            "고객이 제공한 제품 ID를 확인하고 check_warranty 도구로 보증 상태를 조회해요. "
            "보증 상태를 확인한 후, 이슈 분류 단계로 넘어가야 한다고 안내해요."
        ),
        "tools": [check_warranty],
        "next_step": "issue_classify",
    },
    # 2단계: 문제 분류
    "issue_classify": {
        "system_prompt": (
            "당신은 고객 지원팀의 문제 분류 전문가예요. "
            "고객의 문제 설명을 분석해 classify_issue 도구로 문제 유형을 파악해요. "
            "분류 후 해결 단계로 넘어간다고 안내해요."
        ),
        "tools": [classify_issue],
        "next_step": "resolve",
    },
    # 3단계: 해결 처리
    "resolve": {
        "system_prompt": (
            "당신은 고객 지원팀의 해결사예요. "
            "이전 단계에서 수집된 정보를 바탕으로 provide_solution 도구로 해결책을 제시하고, "
            "필요하면 create_support_ticket 도구로 티켓을 생성해요. "
            "고객에게 친절하고 명확하게 최종 답변을 제공해요."
        ),
        "tools": [provide_solution, create_support_ticket],
        "next_step": None,  # 마지막 단계: 더 이상 전환 없음
    },
}

# STEP_CONFIG 정의 완료
for step_name, config in STEP_CONFIG.items():
    tools_names = [t.name for t in config["tools"]]
    next_step = config.get("next_step", "없음")
    print(f"  [{step_name}] 도구: {tools_names} → 다음: {next_step}")

  [warranty_check] 도구: ['check_warranty'] → 다음: issue_classify
  [issue_classify] 도구: ['classify_issue'] → 다음: resolve
  [resolve] 도구: ['provide_solution', 'create_support_ticket'] → 다음: None


## 4. 미들웨어 기반 단일 에이전트 Handoffs (권장 방식)

### @wrap_model_call 미들웨어

`@wrap_model_call`은 LLM 호출 전후에 실행되는 미들웨어예요. Handoffs 패턴에서는 **LLM 호출 직전**에 `current_step`을 확인해 시스템 프롬프트와 도구를 동적으로 교체해요.

```python
@wrap_model_call
def step_switcher(request, call_next):
    # LLM 호출 전: current_step 확인
    step = request.state["current_step"]
    config = STEP_CONFIG[step]
    
    # 시스템 프롬프트와 도구를 이 단계의 설정으로 교체
    modified = request.override(
        system_prompt=config["system_prompt"],
        tools=config["tools"]
    )
    
    # 수정된 설정으로 LLM 호출
    return call_next(modified)
```

> 💡 **핵심 정리**: 미들웨어 방식의 장점은 **에이전트 코드를 수정하지 않고** 행동을 바꿀 수 있다는 점이에요. `STEP_CONFIG`만 편집하면 새로운 단계를 추가하거나 기존 단계의 프롬프트를 수정할 수 있어요. 관심사 분리(Separation of Concerns)의 좋은 예예요.

> ⚠️ **자주 하는 실수**: `request.state`는 읽기 전용이에요. 상태를 변경하려면 반드시 `Command(update={...})`를 통해야 해요. 미들웨어 안에서 직접 상태를 수정하려 하면 오류가 발생해요.

In [ ]:
from langchain.agents.middleware import wrap_model_call

def step_switcher(request, call_next):
    """
    current
    """

    current_step = request.state.get("current_step", "warranty_check")

    if current_step not in STEP_CONFIG:
        return call_next(request)

    step_config = STEP_CONFIG[current_step]

    modified_request = request.override


## 5. Command로 단계 전환하기

에이전트가 현재 단계 작업을 완료한 뒤 다음 단계로 이동하려면 `Command` 객체를 사용해요.

### Command 사용 방법

```python
from langgraph.types import Command

# 다음 단계로 이동: 상태 업데이트 + 노드 이동
return Command(
    update={
        "current_step": "issue_classify",  # 상태에 다음 단계 기록
        "warranty_status": "valid",         # 이번 단계 결과 저장
    },
    goto="support_agent",  # 다시 같은 노드로 (미들웨어가 단계 변경 감지)
)
```

> 🔑 **핵심 개념**: 단일 에이전트 방식에서 `goto`는 자기 자신 노드를 가리켜요. 같은 에이전트가 다시 호출되지만, `current_step`이 변경됐으므로 미들웨어가 다른 프롬프트와 도구를 적용해요. 마치 같은 배우가 다른 대본으로 연기하는 것과 같아요.

### 노드 함수 구조

```python
def support_agent_node(state: SupportState) -> Command:
    # 1. 현재 단계 확인
    # 2. 미들웨어가 적용된 에이전트 실행
    # 3. 결과를 상태에 저장하고 다음 단계로 Command 반환
```

In [ ]:
from langchain.agents import create_agent
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 실행: 미들웨어가 현재 단계의 프롬프트와 도구를 적용해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 그래프 구성 및 실행

단일 에이전트 Handoffs에서 그래프 구조는 매우 단순해요. 노드 하나, 간단한 엣지 구조만 있어요.

```
START → support_agent → END
        (내부에서 Command로 자신을 반복 호출)
```

> 💡 **실무 팁**: 단일 에이전트 방식은 그래프 구조가 단순해서 디버깅과 테스트가 쉬워요. 복잡한 워크플로우도 STEP_CONFIG 설정과 미들웨어로 제어하기 때문에 그래프 코드는 거의 변경되지 않아요.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → support_agent → END (내부에서 Command로 자기 자신을 반복 호출)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 멀티 에이전트 서브그래프 방식

두 번째 구현 방식은 **실제 여러 에이전트가 제어권을 넘기는** 서브그래프 방식이에요.

### 서브그래프 Handoffs 아키텍처

```mermaid
flowchart LR
    U(["사용자"]) --> W
    W(["보증 확인 에이전트"]) -->|"transfer_to_classify"| C
    C(["문제 분류 에이전트"]) -->|"transfer_to_resolve"| R
    R(["해결 에이전트"]) --> FIN(["완료"])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class U input
    class W,C,R process
    class FIN output
```

### 핸드오프 쌍 (Handoff Pair)

서브그래프 방식에서 에이전트 간 제어권 이전은 반드시 두 메시지의 쌍으로 이루어져요:

```
1. AIMessage(tool_calls=[{"name": "transfer_to_classify", ...}])  ← 이전 의도
2. ToolMessage(tool_call_id=..., content="분류 에이전트로 이전")    ← 이전 확인
```

> 🔑 **핵심 개념**: 핸드오프 쌍에서 `AIMessage`만 또는 `ToolMessage`만 전달하면 LLM이 대화 이력을 올바르게 파악하지 못해요. 두 메시지가 반드시 함께 다음 에이전트에게 전달되어야 해요.

> 💡 **실무 팁**: 단일 에이전트 방식(미들웨어)과 서브그래프 방식 중 어떤 걸 선택할까요? 미들웨어 방식은 코드가 간결하고 유지보수가 쉬워요. 서브그래프 방식은 각 에이전트가 완전히 독립적이어야 하거나, 에이전트마다 다른 모델을 사용하고 싶을 때 적합해요.

In [ ]:
from langchain.tools import tool




In [ ]:
from langchain.agents import create_agent

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.messages import AIMessage, ToolMessage
from langgraph.types import Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → warranty_node → classify_node → resolve_node → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# ---------------------------------------------------
# 서브그래프 방식: 그래프 구성 및 컴파일
# ---------------------------------------------------
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 서브그래프 방식: 3개의 노드가 순서대로 연결돼요
subgraph_workflow = StateGraph(SupportState)

# 노드 추가
subgraph_workflow.add_node("warranty_node", warranty_node)
subgraph_workflow.add_node("classify_node", classify_node)
subgraph_workflow.add_node("resolve_node", resolve_node)

# 엣지: START에서 보증 확인으로 시작
# 이후 흐름은 각 노드의 Command가 결정해요
subgraph_workflow.add_edge(START, "warranty_node")
subgraph_workflow.add_edge("resolve_node", END)

# 컴파일
subgraph = subgraph_workflow.compile(checkpointer=MemorySaver())

# 서브그래프 컴파일 완료
#   구조: START → warranty_node → classify_node → resolve_node → END

# 그래프 흐름: START → warranty_node → classify_node → resolve_node → END
# warranty_node: 보증 상태를 확인하고 Command로 classify_node로 핸드오프해요
# classify_node: 문제 유형을 분류하고 Command로 resolve_node로 핸드오프해요
# resolve_node: 해결책을 제시하고 티켓을 생성해요
# 핸드오프 쌍: AIMessage(tool_calls) + ToolMessage(tool_call_id) 페어링 필수
display(Image(subgraph.get_graph().draw_mermaid_png()))

## 8. 두 방식 성능 비교

단일 에이전트 방식과 서브그래프 방식의 LLM 호출 횟수를 비교해봐요.

### 예상 LLM 호출 횟수

| 시나리오 | 단일 에이전트 방식 | 서브그래프 방식 |
|---------|-----------------|----------------|
| 단순 요청 (3단계) | ~3회 (단계별 1회) | ~3회 (에이전트별 1회) |
| 반복 대화 (이력 유지) | ~5회 (이전 컨텍스트 활용) | ~5회 (이전 컨텍스트 활용) |
| 복잡한 다중 도메인 | 7+회 (단계 내 추가 호출) | 7+회 (에이전트 간 교환) |

> 💡 **실무 팁**: 두 방식의 LLM 호출 횟수는 비슷하지만, **단일 에이전트 방식이 대화 이력을 더 효율적으로 유지**해요. 서브그래프 방식에서는 핸드오프 쌍 메시지가 추가로 생성되므로 컨텍스트가 약간 더 커져요.

> 💡 **핵심 정리**: 어떤 방식을 선택할지는 요구사항에 따라 달라요. 미들웨어 방식은 **단순성과 유지보수성**, 서브그래프 방식은 **독립성과 확장성**에서 유리해요. 실무에서는 미들웨어 방식을 먼저 시도하고, 에이전트 간 완전한 분리가 필요할 때 서브그래프로 전환하는 것을 권장해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. 실습: 단계 추가해보기

아래 TODO 블록을 완성해 **고객 만족도 조사** 단계를 3단계 워크플로우 끝에 추가해보세요.

> 💡 **힌트**: 단일 에이전트 방식에서는 `STEP_CONFIG`에 항목 하나만 추가하면 돼요. `resolve` 단계의 `next_step`을 새 단계로 바꾸고, 새 단계의 `next_step`은 `None`으로 설정하세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 고객 만족도 조사 단계를 추가해보세요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Handoffs 패턴**: Supervisor 없이 상태 변수(`current_step`)로 에이전트 행동을 직접 제어해요. LLM 라우팅 비용 없이 예측 가능한 워크플로우를 만들 수 있어요.

- **단일 에이전트 + 미들웨어 (권장)**: `@wrap_model_call`이 `current_step`을 보고 LLM 호출 전에 프롬프트와 도구를 교체해요. `STEP_CONFIG` 딕셔너리에 단계 설정을 분리해 유지보수가 쉬워요.

- **STEP_CONFIG 패턴**: 각 단계의 시스템 프롬프트, 도구, 다음 단계를 딕셔너리로 중앙 관리해요. 새 단계 추가 시 딕셔너리에 항목 하나만 추가하면 돼요.

- **ToolMessage 페어링**: 에이전트 간 핸드오프 시 반드시 `AIMessage(tool_calls)`와 `ToolMessage(tool_call_id)`를 쌍으로 전달해요. `tool_call_id` 불일치는 대화 이력 오류의 주요 원인이에요.

- **Command로 단계 전환**: `Command(update={"current_step": "다음단계"}, goto="node_name")`으로 상태를 업데이트하면서 다음 노드로 이동해요. 단일 에이전트 방식에서는 자기 자신 노드를 `goto`로 지정해 재호출해요.

## 다음 노트북 예고

다음 `06-Router-Pattern.ipynb`에서는 **Router 패턴**을 배워요. Handoffs가 순차적 인계라면, Router는 **분류기 LLM이 질문을 병렬로 여러 에이전트에 분배**해요. `with_structured_output`으로 분류 결과를 구조화하고, `Send` API로 동적으로 여러 에이전트에 병렬 전송한 뒤 결과를 합산해요. 경량 분류기 + 특화 에이전트 조합으로 정확도와 비용을 동시에 잡는 패턴이에요.